M

In [8]:
# ============================================================
# TEST API CONNECTION - LOCAL JUPYTER
# ============================================================

import requests

API_KEY = "futlandfory"

try:
    r = requests.get(
        "https://api.europeana.eu/record/v2/search.json",
        params={
            "wskey":   API_KEY,
            "query":   "painting",
            "rows":    1,
            "profile": "minimal"
        },
        timeout=30
    )
    print(f"Status: {r.status_code}")
    print(f"Total results: {r.json().get('totalResults', 'N/A')}")
    print("SUCCESS! We can now collect data!")
except Exception as e:
    print(f"Failed: {e}")

Status: 200
Total results: 489280
SUCCESS! We can now collect data!


# Notebook 01: Data Collection from Europeana API

**Project:** Who Makes the Canon? Gender Representation in European Painting Collections  
**Author:** your name  
**Date:** June 2026

## Purpose
This notebook collects painting records from the Europeana API.
We query by country to collect 50,000+ records with creator metadata.

## Output
- `../data/raw/europeana_raw.csv` — all collected records
- `../data/raw/collection_summary.csv` — records per country

In [10]:
# ============================================================
# IMPORTS AND CONFIGURATION
# ============================================================

import requests
import pandas as pd
import time
import os

# --- YOUR API KEY ---
API_KEY = "futlandfory"

BASE_URL = "https://api.europeana.eu/record/v2/search.json"
RECORDS_PER_PAGE = 100
MAX_PER_COUNTRY  = 10000

os.makedirs("../data/raw", exist_ok=True)
OUTPUT_FILE = "../data/raw/europeana_raw.csv"

COUNTRIES = [
    "france", "germany", "netherlands", "italy", "spain",
    "united_kingdom", "belgium", "austria", "sweden", "denmark",
    "poland", "portugal", "czech_republic", "hungary", "greece",
    "finland", "norway", "switzerland", "croatia", "slovakia"
]

print(f"Ready to query {len(COUNTRIES)} countries")
print(f"Maximum possible records: {len(COUNTRIES) * MAX_PER_COUNTRY:,}")

Ready to query 20 countries
Maximum possible records: 200,000


In [11]:
# ============================================================
# FUNCTION: get_total_results
# ============================================================

def get_total_results(country):
    params = {
        "wskey":   API_KEY,
        "query":   "painting",
        "qf":      [
                    "TYPE:IMAGE",
                    "proxy_dc_creator:*",
                    f"COUNTRY:{country}"
                   ],
        "rows":    1,
        "profile": "minimal",
    }
    try:
        r = requests.get(BASE_URL, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        return data.get("totalResults", 0)
    except Exception as e:
        print(f"  Could not get total for {country}: {e}")
        return 0


# ============================================================
# FUNCTION: extract_record_fields
# ============================================================

def extract_record_fields(item, country_queried):
    
    def get_first(field):
        value = item.get(field, None)
        if isinstance(value, list) and len(value) > 0:
            return str(value[0])
        elif value is not None:
            return str(value)
        return None
    
    return {
        "id":               item.get("id", None),
        "title":            get_first("title"),
        "creator":          get_first("dcCreator"),
        "date":             get_first("year"),
        "country":          get_first("country"),
        "country_queried":  country_queried,
        "provider":         get_first("dataProvider"),
        "rights":           get_first("rights"),
        "description":      get_first("dcDescription"),
        "medium":           get_first("dcFormat"),
        "language":         get_first("language"),
        "thumbnail":        item.get("edmPreview", [None])[0] 
                            if item.get("edmPreview") else None,
    }


# ============================================================
# FUNCTION: collect_country_records
# ============================================================

def collect_country_records(country, max_records=MAX_PER_COUNTRY):
    
    country_records = []
    start_index = 1
    
    while len(country_records) < max_records:
        
        params = {
            "wskey":   API_KEY,
            "query":   "painting",
            "qf":      [
                        "TYPE:IMAGE",
                        "proxy_dc_creator:*",
                        f"COUNTRY:{country}"
                       ],
            "start":   start_index,
            "rows":    RECORDS_PER_PAGE,
            "profile": "rich",
        }
        
        try:
            r = requests.get(BASE_URL, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()
            items = data.get("items", [])
            
            if not items:
                break
            
            for item in items:
                record = extract_record_fields(item, country)
                country_records.append(record)
            
            start_index += RECORDS_PER_PAGE
            
            if len(items) < RECORDS_PER_PAGE:
                break
                
            time.sleep(0.5)
            
        except Exception as e:
            print(f"    Error at start={start_index}: {e}")
            time.sleep(2)
            break
    
    return country_records

print("All functions defined successfully!")

All functions defined successfully!


In [12]:
# ============================================================
# MAIN COLLECTION LOOP
# WARNING: This will take 30-60 minutes to complete
# Do NOT close the browser while it is running
# ============================================================

all_records = []
country_summary = []

print("Starting collection by country...\n")
print(f"{'Country':<20} {'Available':>10} {'Collected':>10}")
print("-" * 42)

for country in COUNTRIES:
    
    total_available = get_total_results(country)
    time.sleep(0.3)
    
    if total_available == 0:
        print(f"{country:<20} {'0':>10} {'skipped':>10}")
        continue
    
    records = collect_country_records(country)
    all_records.extend(records)
    
    country_summary.append({
        "country": country,
        "available": total_available,
        "collected": len(records)
    })
    
    print(f"{country:<20} {total_available:>10,} {len(records):>10,}")
    print(f"  Running total: {len(all_records):,} records")
    
    time.sleep(1)


# ============================================================
# SAVE RESULTS
# ============================================================

print(f"\n{'='*42}")
print(f"COLLECTION COMPLETE")
print(f"{'='*42}")
print(f"Total records collected: {len(all_records):,}")

df_summary = pd.DataFrame(country_summary)
print(f"\nCountry breakdown:")
print(df_summary.to_string(index=False))
df_summary.to_csv("../data/raw/collection_summary.csv", index=False)

df_raw = pd.DataFrame(all_records)

before_dedup = len(df_raw)
df_raw = df_raw.drop_duplicates(subset=["id"])
after_dedup = len(df_raw)
print(f"\nDuplicates removed: {before_dedup - after_dedup:,}")
print(f"Final unique records: {after_dedup:,}")

print(f"\nMissing values per column:")
print(df_raw.isnull().sum())

print(f"\nFirst 5 records:")
print(df_raw.head())

df_raw.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
print(f"\nRaw data saved to: {OUTPUT_FILE}")
print(f"Notebook 01 complete!")

Starting collection by country...

Country               Available  Collected
------------------------------------------
    Error at start=1001: 400 Client Error: Bad Request for url: https://api.europeana.eu/record/v2/search.json?wskey=futlandfory&query=painting&qf=TYPE%3AIMAGE&qf=proxy_dc_creator%3A%2A&qf=COUNTRY%3Afrance&start=1001&rows=100&profile=rich
france                   17,043      1,000
  Running total: 1,000 records
    Error at start=1001: 400 Client Error: Bad Request for url: https://api.europeana.eu/record/v2/search.json?wskey=futlandfory&query=painting&qf=TYPE%3AIMAGE&qf=proxy_dc_creator%3A%2A&qf=COUNTRY%3Agermany&start=1001&rows=100&profile=rich
germany                 121,909      1,000
  Running total: 2,000 records
    Error at start=1001: 400 Client Error: Bad Request for url: https://api.europeana.eu/record/v2/search.json?wskey=futlandfory&query=painting&qf=TYPE%3AIMAGE&qf=proxy_dc_creator%3A%2A&qf=COUNTRY%3Anetherlands&start=1001&rows=100&profile=rich
netherla

In [13]:
# ============================================================
# IMPROVED COLLECTION: Get past 1000 record limit
# Strategy: Query by YEAR RANGE within each country
# This gets us past Europeana's 1000 record pagination limit
# ============================================================

all_records = []
country_summary = []

# Year ranges to split queries into smaller chunks
YEAR_RANGES = [
    ("1000", "1599"),
    ("1600", "1699"),
    ("1700", "1799"),
    ("1800", "1849"),
    ("1850", "1899"),
    ("1900", "1949"),
    ("1950", "2000"),
]

print("Starting improved collection by country + year range...\n")

for country in COUNTRIES:
    country_total = 0
    
    print(f"\nCollecting: {country.upper()}")
    
    for year_start, year_end in YEAR_RANGES:
        
        start_index = 1
        chunk_records = []
        
        while True:
            params = {
                "wskey":   API_KEY,
                "query":   f"painting AND YEAR:[{year_start} TO {year_end}]",
                "qf":      [
                            "TYPE:IMAGE",
                            "proxy_dc_creator:*",
                            f"COUNTRY:{country}"
                           ],
                "start":   start_index,
                "rows":    100,
                "profile": "rich",
            }
            
            try:
                r = requests.get(BASE_URL, params=params, timeout=30)
                
                if r.status_code == 400:
                    break
                    
                r.raise_for_status()
                data = r.json()
                items = data.get("items", [])
                
                if not items:
                    break
                
                for item in items:
                    record = extract_record_fields(item, country)
                    chunk_records.append(record)
                
                start_index += 100
                
                if len(items) < 100 or start_index > 1000:
                    break
                
                time.sleep(0.3)
                
            except Exception as e:
                break
        
        if chunk_records:
            all_records.extend(chunk_records)
            country_total += len(chunk_records)
            print(f"  {year_start}-{year_end}: {len(chunk_records)} records")
        
        time.sleep(0.5)
    
    country_summary.append({
        "country": country,
        "collected": country_total
    })
    print(f"  Total for {country}: {country_total:,} records")
    print(f"  Running total: {len(all_records):,} records")
    time.sleep(1)


# ============================================================
# SAVE IMPROVED RESULTS
# ============================================================

print(f"\n{'='*42}")
print(f"COLLECTION COMPLETE")
print(f"{'='*42}")
print(f"Total records collected: {len(all_records):,}")

df_summary = pd.DataFrame(country_summary)
print(f"\nCountry breakdown:")
print(df_summary.to_string(index=False))
df_summary.to_csv("../data/raw/collection_summary_v2.csv", index=False)

df_raw = pd.DataFrame(all_records)

before_dedup = len(df_raw)
df_raw = df_raw.drop_duplicates(subset=["id"])
after_dedup = len(df_raw)

print(f"\nDuplicates removed: {before_dedup - after_dedup:,}")
print(f"Final unique records: {after_dedup:,}")

df_raw.to_csv("../data/raw/europeana_raw_v2.csv", index=False, encoding="utf-8")
print(f"\nData saved to: ../data/raw/europeana_raw_v2.csv")
print(f"Notebook 01 complete!")

Starting improved collection by country + year range...


Collecting: FRANCE
  1600-1699: 14 records
  1700-1799: 152 records
  1800-1849: 39 records
  1850-1899: 151 records
  1900-1949: 1000 records
  1950-2000: 1000 records
  Total for france: 2,356 records
  Running total: 2,356 records

Collecting: GERMANY
  1000-1599: 96 records
  1600-1699: 84 records
  1700-1799: 175 records
  1800-1849: 174 records
  1850-1899: 302 records
  1900-1949: 765 records
  1950-2000: 329 records
  Total for germany: 1,925 records
  Running total: 4,281 records

Collecting: NETHERLANDS
  1000-1599: 4 records
  1600-1699: 21 records
  1700-1799: 43 records
  1800-1849: 20 records
  1850-1899: 48 records
  1900-1949: 105 records
  1950-2000: 329 records
  Total for netherlands: 570 records
  Running total: 4,851 records

Collecting: ITALY
  1000-1599: 13 records
  1600-1699: 46 records
  1700-1799: 494 records
  1800-1849: 49 records
  1850-1899: 491 records
  1900-1949: 791 records
  1950-2000: 591 rec

In [ ]:
import os

# Find where we are right now
print("Current location:")
print(os.getcwd())

# Look for any CSV files on the computer
print("\nLooking for CSV files...")
for root, dirs, files in os.walk(os.path.expanduser("~")):
    for file in files:
        if file.endswith(".csv") and "europeana" in file.lower():
            print(os.path.join(root, file))

Current location:
C:\Users\bfde3\Documents

Looking for CSV files...
